<a href="https://colab.research.google.com/github/Bpatnaik470/Bpatnaik470/blob/main/live_class_323.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier


import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [29]:
x, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=6,
    n_redundant= 2,
    n_classes = 2,
    random_state=42
)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)
print("Training Shape:", x_train.shape)
print("Testing Shape", x_test.shape)

Training Shape: (750, 10)
Testing Shape (250, 10)


In [30]:
log_model = LogisticRegression(max_iter=1000)
tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
svm_model = SVC(probability=True, random_state=42)

model = {
    "Logistic Regression": log_model,
    "Decision Tree": tree_model,
    "Support Vector Machine": svm_model
}
baseline_results = []
for name, model in model.items():
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    accuracy = accuracy_score(y_test, pred)
    baseline_results.append((name, accuracy))

    print(f"{name} Accuracy: {accuracy:.4f}")

Logistic Regression Accuracy: 0.8160
Decision Tree Accuracy: 0.8360
Support Vector Machine Accuracy: 0.9040


In [31]:
baseline_df = pd.DataFrame(baseline_results, columns=["Model", "Accuracy"])
baseline_df.sort_values(by="Accuracy", ascending=False)


,Model,Accuracy
2,Support Vector Machine,0.904
1,Decision Tree,0.836
0,Logistic Regression,0.816


In [32]:
hard_voter = VotingClassifier(
    estimators=[
        ("lr", log_model),
        ("dt", tree_model),
        ("svm", svm_model)
    ],
    voting="hard"
)
hard_voter.fit(x_train, y_train)
hard_pred = hard_voter.predict(x_test)
hard_accuracy = accuracy_score(y_test, hard_pred)
print(f"Hard Voting Accuracy: {hard_accuracy:.4f}")

Hard Voting Accuracy: 0.8640


In [33]:
soft_voter = VotingClassifier(
    estimators=[
        ("lr", log_model),
        ("dt", tree_model),
        ("svm", svm_model)
    ],
    voting="soft"
)
soft_voter.fit(x_train, y_train)
soft_pred = soft_voter.predict(x_test)
soft_accuracy = accuracy_score(y_test, soft_pred)
print(f"Soft Voting Accuracy: {soft_accuracy:.4}")

Soft Voting Accuracy: 0.86


In [34]:
log_model.fit(x_train, y_train)
tree_model.fit(x_train, y_train)
svm_model.fit(x_train, y_train)

log_prob = log_model.predict_proba(x_test)[:, 1]
tree_prob = tree_model.predict_proba(x_test)[:, 1]
svm_prob = svm_model.predict_proba(x_test)[:, 1]

avg_prob = (log_prob + tree_prob + svm_prob) / 3
avg_pred  = (avg_prob >= 0.5).astype(int)

manual_avg_acc = accuracy_score(y_test, avg_pred)
print(f"Manual Average Accuracy: {manual_avg_acc:.4f}")

Manual Average Accuracy: 0.8600


In [41]:
comparison_prob = pd.DataFrame({
    "Logistic_Prob_Class1": log_prob[:10],
    "Tree_Prob_Class1": tree_prob[:10],
    "SVM_Prob_Class1": svm_prob[:10],
    "Avergae_Prob_Class1": avg_prob[:10],
    "Final_Prediction": avg_pred[:10],
    "Actual_Class1": y_test[:10]
})
comparison_prob

,Logistic_Prob_Class1,Tree_Prob_Class1,SVM_Prob_Class1,Avergae_Prob_Class1,Final_Prediction,Actual_Class1
0,0.830715,0.300000,0.096967,0.409227,0,0
1,0.254167,0.375000,0.431111,0.353426,0,1
2,0.078192,0.023256,0.014100,0.038516,0,0
3,0.488917,0.961207,0.737896,0.729340,1,1
4,0.834241,0.961207,0.930154,0.908534,1,0
5,0.000800,0.023256,0.050039,0.024698,0,0
6,0.029789,0.023256,0.002995,0.018680,0,0
7,0.566226,1.000000,0.019426,0.528551,1,0
8,0.980943,0.961207,0.969800,0.970650,1,1
9,0.962923,0.770270,0.921695,0.884963,1,1


In [36]:
weights = np.array([0.5, 0.2, 0.3])

weighted_prob = (
    weights[0] * log_prob +
    weights[1] * tree_prob +
    weights[2] * svm_prob
)

weighted_pred = (weighted_prob >= 0.5).astype(int)
weighted_acc = accuracy_score(y_test, weighted_pred)

print("Weighted Averaging Accuracy:", round(weighted_acc, 4))

Weighted Averaging Accuracy: 0.844


In [40]:
single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(x_train, y_train)
single_tree_pred = single_tree.predict(x_test)
single_tree_acc = accuracy_score(y_test, single_tree_pred)

bagging_model = BaggingClassifier(
    estimator = DecisionTreeClassifier(),
    n_estimators = 50, # Corrected parameter name
    random_state = 42,
    oob_score = True
)

bagging_model.fit(x_train, y_train)
bagging_pred = bagging_model.predict(x_test)
bagging_acc = accuracy_score(y_test, bagging_pred)

print("Single Tree Accuracy:", round(single_tree_acc, 4))
print("Bagging Accuracy:", round(bagging_acc, 4))
print("Bagging OOB Score:", round(bagging_model.oob_score_, 4))

Single Tree Accuracy: 0.832
Bagging Accuracy: 0.876
Bagging OOB Score: 0.9053


In [39]:
rf_model = RandomForestClassifier(
    n_estimators=100,
       random_state=42
)

rf_model.fit(x_train, y_train)
rf_pred = rf_model.predict(x_test)
rf_acc = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", round(rf_acc, 4))

Random Forest Accuracy: 0.872


In [45]:
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate = 0.1,
       random_state=42
)

gb_model.fit(x_train, y_train)
gb_pred = gb_model.predict(x_test)
gb_acc = accuracy_score(y_test, gb_pred)

print("Gradient Boosting Accuracy:", round(gb_acc, 4))

Gradient Boosting Accuracy: 0.868


In [48]:
stack_model = StackingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=1000)),
        ('knn', KNeighborsClassifier(n_neighbors=5)),
        ('tree', DecisionTreeClassifier(max_depth=4, random_state=42))

    ],
        final_estimator=LogisticRegression()
)
stack_model.fit(x_train, y_train)
stack_model_pred = stack_model.predict(x_test)
stack_model_acc = accuracy_score(y_test, stack_model_pred)
print("Stacking Accuracy:", round(stack_model_acc, 4))

Stacking Accuracy: 0.924


In [53]:
results = [
    ("Logistic Regression", baseline_df.loc[baseline_df["Model"] == "Logistic Regression", "Accuracy"].values[0]),
    ("Decision Tree", baseline_df.loc[baseline_df["Model"] == "Decision Tree", "Accuracy"].values[0]),
    ("Support Vector Machine", baseline_df.loc[baseline_df["Model"] == "Support Vector Machine", "Accuracy"].values[0]),
    ("Hard Voting", hard_accuracy),
    ("Soft Voting", soft_accuracy),
    ("Manual Average", manual_avg_acc),
    ("Weighted Averaging", weighted_acc),
    ("Single Tree", single_tree_acc),
    ("Bagging", bagging_acc),
    ("Random Forest", rf_acc),
    ("AdaBoost", ada_acc),
    ("Gradient Boosting", gb_acc),
    ("Stacking", stack_model_acc)
]
results_df = pd.DataFrame(results, columns=["Model", "Accuracy"]).sort_values("Accuracy", ascending=False)
display(results_df.reset_index(drop=True))

,Model,Accuracy
0,Stacking,0.924
1,Support Vector Machine,0.904
2,Bagging,0.876
3,Random Forest,0.872
4,Gradient Boosting,0.868
5,Hard Voting,0.864
6,Manual Average,0.860
7,Soft Voting,0.860
8,Weighted Averaging,0.844
9,Decision Tree,0.836


In [51]:
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=100,
    random_state=42
)

ada_model.fit(x_train, y_train)
ada_pred = ada_model.predict(x_test)
ada_acc = accuracy_score(y_test, ada_pred)

print("AdaBoost Accuracy:", round(ada_acc, 4))

AdaBoost Accuracy: 0.764
